# Diabeteksen etenemisen ennustaminen

## 1. Johdanto

Diabetes on pitkäaikaissairaus, jonka eteneminen vaihtelee potilaiden välillä. Potilaan lähtötilanteessa mitattujen terveystietojen avulla voidaan pyrkiä ennustamaan sairauden myöhempää etenemistä.

Tämän projektin tavoitteena on tutkia, voidaanko diabeteksen etenemistä ennustaa potilaan lähtötilanteessa mitattujen terveystietojen perusteella. Projektissa verrataan lineaarista regressiota ja Random Forest -regressiota.

**Tutkimuskysymys:** Voidaanko diabeteksen etenemistä ennustaa potilaan lähtötilanteessa mitattujen terveystietojen perusteella, ja kumpi tutkituista regressiomenetelmistä saavuttaa pienemmän validaatiovirheen?

Luvussa 2 määritellään koneoppimisongelma ja käytetty aineisto. Luvussa 3 kuvataan piirteiden valinta, molemmat regressiomenetelmät, häviöfunktio ja validointi. Luvussa 4 vertaillaan tuloksia ja arvioidaan valittu malli testiaineistolla. Luvussa 5 esitetään johtopäätökset ja parannusmahdollisuudet. Lopuksi kuvataan tekoälyn käyttö sekä esitetään lähteet ja koodiliite.

In [14]:
# Tarvittavien kirjastojen importti

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## 2. Ongelman määrittely

Diabeteksen etenemisen ennustaminen muotoillaan ohjatun oppimisen regressio-ongelmaksi, koska ennustettava kohdemuuttuja on jatkuva numeerinen arvo.

Yksi datapiste vastaa yhtä diabetespotilasta. Mallin syötteenä käytetään 10 potilaan lähtötilanteessa mitattua ominaisuuspiirrettä: ikää, sukupuolta, painoindeksiä (BMI), keskimääräistä verenpainetta sekä kuutta veriseerumista mitattua muuttujaa.

Ennustettava kohdemuuttuja (`target`) on numeerinen arvo, joka kuvaa diabeteksen etenemistä vuoden kuluttua lähtötilanteesta. Projektissa käytetään scikit-learn-kirjaston Diabetes-aineistoa [1].

## 3. Menetelmät

### 3.1 Aineisto

Aineisto sisältää 442 datapistettä ja 10 potilaan lähtötilanteessa mitattua ominaisuuspiirrettä. Aineisto ladattiin alkuperäisillä skaalaamattomilla arvoilla (`scaled=False`), jotta muuttujien arvot ovat helposti tulkittavissa.

Aineistossa ei ole puuttuvia arvoja, joten puuttuvien arvojen käsittelyä ei tarvita.

In [15]:
# Ladataan Diabetes aineisto DataFramena

diabetes = load_diabetes(as_frame=True, scaled=False)

# Muodostetaan featuret ja kohdemuuttujan sisältävä DataFrame
df = diabetes.frame


# Näytetään ainoston viisi ensimmäistä riviä
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0,135.0


In [16]:
# Aineiston koko
df.shape

(442, 11)

In [17]:
# Tarkistetaan puuttuvat arvot
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

### 3.2 Piirteiden valinta

Molempien mallien syötteenä käytetään kaikkia aineiston 10 ominaisuuspiirrettä. Kohdemuuttuja (`target`) erotetaan syötepiirteistä ja sitä käytetään mallin ennustettavana muuttujana.

Kaikki saatavilla olevat ominaisuuspiirteet säilytetään, koska tavoitteena on tutkia niiden yhteistä kykyä ennustaa diabeteksen etenemistä. Erillistä piirteiden karsintaa ei tehdä.

In [18]:
# Erotetaan syötepiirteet X ja ennustettava kohdemuuttuja y

X = df.drop(columns="target")
y = df["target"]

print("Piirteiden koko:", X.shape)
print("Kohdemuuttujan koko:", y.shape)

Piirteiden koko: (442, 10)
Kohdemuuttujan koko: (442,)


### 3.3 Lineaarinen regressio

Projektin ensimmäiseksi koneoppimismenetelmäksi valitaan lineaarinen regressio. Mallin ennuste muodostuu vakiotermistä ja syötepiirteiden painotetusta summasta. Malli olettaa siis piirteiden ja ennusteen välisen yhteyden lineaariseksi [2].

Lineaarinen regressio soveltuu jatkuvan numeerisen kohdemuuttujan ennustamiseen useiden syötepiirteiden perusteella. Se valitaan yksinkertaiseksi ja helposti tulkittavaksi vertailukohdaksi joustavammalle Random Forest -mallille. Toteutuksessa käytetään scikit-learnin `LinearRegression`-luokkaa oletusasetuksilla.

### 3.4 Random Forest -regressio

Projektin toiseksi koneoppimismenetelmäksi valitaan Random Forest -regressio. Menetelmä muodostaa useita päätöspuita koulutusaineistosta palauttaen poimituista satunnaisotoksista ja yhdistää puiden ennusteet keskiarvoksi. Useiden puiden yhdistäminen vähentää yksittäisen päätöspuun ennusteiden vaihtelua [3].

Random Forest valittiin vertailumenetelmäksi, koska se pystyy mallintamaan epälineaarisia yhteyksiä piirteiden ja diabeteksen etenemistä kuvaavan kohdemuuttujan välillä. Näin voidaan tutkia, riittääkö yksinkertainen lineaarinen malli vai parantaako joustavampi malli ennustustarkkuutta.

Toteutuksessa käytetään `RandomForestRegressor`-luokkaa, 200 päätöspuuta ja satunnaissiementä 42. Muut asetukset jätetään oletusarvoihin: puiden syvyyttä ei rajoiteta ja lehden havaintojen vähimmäismäärä on yksi. Hyperparametreja ei optimoida tässä kokeessa.

### 3.5 Häviöfunktio

Molempien menetelmien suorituskyvyn arviointiin käytetään keskimääräistä neliövirhettä (Mean Squared Error, MSE). Se lasketaan todellisten ja ennustettujen arvojen erotusten neliöiden keskiarvona. Pienempi MSE tarkoittaa pienempää ennustevirhettä. MSE soveltuu jatkuvan kohdemuuttujan arviointiin ja painottaa suuria ennustevirheitä.

Lineaarinen regressio opetetaan minimoimalla koulutusaineiston neliövirheiden summa [2]. Random Forestin puiden jakokriteerinä käytetään oletusarvoista `squared_error`-kriteeriä, jolla pyritään vähentämään kohdemuuttujan vaihtelua solmujen sisällä [3]. Saman MSE-mittarin käyttäminen mallien arvioinnissa mahdollistaa niiden suoran vertailun.

### 3.6 Mallin validointi

Aineisto jaetaan satunnaisesti koulutus-, validaatio- ja testiaineistoihin likimääräisessä suhteessa 70 % / 15 % / 15 %. Koulutusaineistossa on 309, validaatioaineistossa 66 ja testiaineistossa 67 datapistettä.

Jako tehdään kahdessa vaiheessa `train_test_split`-funktiolla. Ensin 70 % havainnoista erotetaan koulutusaineistoksi. Jäljelle jäävät havainnot jaetaan puoliksi validaatio- ja testiaineistoihin. Molemmissa vaiheissa käytetään satunnaissiementä 42, jotta jako on toistettavissa.

Koulutusaineistoa käytetään molempien mallien opettamiseen. Lopullinen malli valitaan pienemmän validaatio-MSE:n perusteella. Testiaineisto pidetään erillään koulutuksesta ja mallin valinnasta, ja sitä käytetään vasta valitun mallin lopulliseen arviointiin.

Koulutusaineistolle varataan suurin osa havainnoista, jotta mallin opettamiseen on riittävästi dataa; jäljelle jäävä aineisto jaetaan suunnilleen tasan validaatio- ja testiaineistoihin. Samojen osajoukkojen käyttäminen molemmille malleille tekee vertailusta johdonmukaisen.

In [19]:
# Jaetaan aineisto 70 % koulutusaineistoon ja 30 % väliaikaiseen aineistoon
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Jaetaan jäljelle jäävä aineisto puoliksi validaatio- ja testiaineistoihin
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Koulutusaineisto:", X_train.shape)
print("Validaatioaineisto:", X_val.shape)
print("Testiaineisto:", X_test.shape)

Koulutusaineisto: (309, 10)
Validaatioaineisto: (66, 10)
Testiaineisto: (67, 10)


## 4. Tulokset

In [20]:
# Luodaan ja opetetaan mallit
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
# Ennusteet koulutus- ja validaatioaineistoille
linear_train_pred = linear_model.predict(X_train)
linear_val_pred = linear_model.predict(X_val)

rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)

In [22]:
# Lasketaan MSE kummallekin mallille
linear_train_mse = mean_squared_error(y_train, linear_train_pred)
linear_val_mse = mean_squared_error(y_val, linear_val_pred)

rf_train_mse = mean_squared_error(y_train, rf_train_pred)
rf_val_mse = mean_squared_error(y_val, rf_val_pred)

print("Lineaarinen regressio")
print("Koulutus MSE:", linear_train_mse)
print("Validointi MSE:", linear_val_mse)

print("\nRandom Forest")
print("Koulutus MSE:", rf_train_mse)
print("Validointi MSE:", rf_val_mse)

Lineaarinen regressio
Koulutus MSE: 2924.046379072639
Validointi MSE: 2499.3212346817445

Random Forest
Koulutus MSE: 475.0265049352751
Validointi MSE: 2702.5558833333334


### 4.1 Mallien vertailu

| Menetelmä | Koulutus-MSE | Validaatio-MSE |
| --- | ---: | ---: |
| Lineaarinen regressio | 2924,05 | 2499,32 |
| Random Forest -regressio | 475,03 | 2702,56 |

Random Forest saavutti huomattavasti pienemmän virheen koulutusaineistolla, mutta sen validaatiovirhe oli lineaarista regressiota suurempi. Suuri ero Random Forestin koulutus- ja validaatiovirheen välillä viittaa mallin ylisovittamiseen koulutusaineistoon.

Lineaarisen regression validaatiovirhe oli pienempi, joten se valitaan projektin lopulliseksi malliksi. Sen suorituskyky validaatioaineistolla oli parempi, mikä viittaa parempaan yleistymiskykyyn tässä aineistojaossa. Lineaarisen regression validaatiovirhe oli koulutusvirhettä pienempi; eri osajoukkojen satunnainen koostumus voi selittää tätä eroa.

In [23]:
# Arvioidaan lopullinen malli testiaineistolla
linear_test_pred = linear_model.predict(X_test)
linear_test_mse = mean_squared_error(y_test, linear_test_pred)

print("Lineaarisen regression test MSE:", linear_test_mse)

Lineaarisen regression test MSE: 3139.3683430474566


### 4.2 Lopullisen mallin testitulos

Lopulliseksi malliksi valitun lineaarisen regression testiaineiston MSE oli 3139,37. Testivirhe on jonkin verran suurempi kuin validaatioaineiston MSE 2499,32, mutta samaa suuruusluokkaa. Testiaineistoa ei käytetty mallien kouluttamiseen tai mallin valintaan, joten tulos toimii lopullisena arviona mallin suorituskyvystä uudella aineistolla.

## 5. Johtopäätökset

Projektissa verrattiin lineaarista regressiota ja Random Forest -regressiota diabeteksen etenemistä kuvaavan jatkuvan arvon ennustamisessa. Mallien suorituskykyä arvioitiin keskimääräisellä neliövirheellä (MSE).

Random Forest saavutti selvästi pienemmän koulutusvirheen kuin lineaarinen regressio, mutta sen validaatiovirhe oli suurempi. Tämä viittaa siihen, että Random Forest ylisovitti koulutusaineistoa. Lineaarisen regression validaatiovirhe oli pienempi, joten se valittiin lopulliseksi malliksi. Lopullisen lineaarisen regressiomallin testiaineiston MSE oli 3139,37.

Lineaarinen regressio pärjäsi tässä vertailussa Random Forestia paremmin, mutta pelkän MSE-arvon perusteella ei voida päätellä ennusteiden riittävyyttä käytännön käyttötarkoitukseen ilman vertailutasoa tai hyväksyttävän virheen määrittelyä. Testivirhe oli validaatiovirhettä suurempi. Pienen aineiston yksittäinen satunnaisjako tuo tuloksiin epävarmuutta, joten tulos ei osoita menetelmien yleistä paremmuusjärjestystä.

Mallia voitaisiin jatkossa parantaa esimerkiksi tutkimalla erilaisia piirteiden valintatapoja, kokeilemalla muita regressiomenetelmiä sekä optimoimalla mallien hyperparametreja. Random Forestin tapauksessa ylisovittamista voitaisiin pyrkiä vähentämään esimerkiksi rajoittamalla puiden syvyyttä tai kasvattamalla lehteen vaadittavien havaintojen vähimmäismäärää. Myös suurempi aineisto voisi parantaa mallien yleistymiskykyä. Jatkossa mallien valintaa voitaisiin arvioida ristiinvalidoinnilla koulutus- ja validaatioaineiston sisällä. Testiaineisto tulisi edelleen pitää erillään kehityksestä.

## 6. Generatiivisen tekoälyn käyttö

Projektissa käytettiin OpenAI ChatGPT:tä projektiohjeiden tulkitsemisen, notebookin rakenteen suunnittelun, tekstin muotoilun ja Python-koodin tarkistamisen tukena. ChatGPT:n rooli oli neuvova.

## 7. Lähteet

[1] Scikit-learn Developers. *load_diabetes*. Scikit-learnin dokumentaatio. https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html

[2] Scikit-learn Developers. *LinearRegression*. Scikit-learnin dokumentaatio. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

[3] Scikit-learn Developers. *RandomForestRegressor*. Scikit-learnin dokumentaatio. https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

## 8. Liite: koodi

Alla on notebookin suoritettavien koodisolujen sisältö alkuperäisessä järjestyksessä.

```python
# Tarvittavien kirjastojen importti

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Ladataan Diabetes aineisto DataFramena

diabetes = load_diabetes(as_frame=True, scaled=False)

# Muodostetaan featuret ja kohdemuuttujan sisältävä DataFrame
df = diabetes.frame


# Näytetään ainoston viisi ensimmäistä riviä
df.head()

# Aineiston koko
df.shape

# Tarkistetaan puuttuvat arvot
df.isnull().sum()

# Erotetaan syötepiirteet X ja ennustettava kohdemuuttuja y

X = df.drop(columns="target")
y = df["target"]

print("Piirteiden koko:", X.shape)
print("Kohdemuuttujan koko:", y.shape)

# Jaetaan aineisto 70 % koulutusaineistoon ja 30 % väliaikaiseen aineistoon
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Jaetaan jäljelle jäävä aineisto puoliksi validaatio- ja testiaineistoihin
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print("Koulutusaineisto:", X_train.shape)
print("Validaatioaineisto:", X_val.shape)
print("Testiaineisto:", X_test.shape)

# Luodaan ja opetetaan mallit
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)
rf_model.fit(X_train, y_train)

# Ennusteet koulutus- ja validaatioaineistoille
linear_train_pred = linear_model.predict(X_train)
linear_val_pred = linear_model.predict(X_val)

rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)

# Lasketaan MSE kummallekin mallille
linear_train_mse = mean_squared_error(y_train, linear_train_pred)
linear_val_mse = mean_squared_error(y_val, linear_val_pred)

rf_train_mse = mean_squared_error(y_train, rf_train_pred)
rf_val_mse = mean_squared_error(y_val, rf_val_pred)

print("Lineaarinen regressio")
print("Koulutus MSE:", linear_train_mse)
print("Validointi MSE:", linear_val_mse)

print("\nRandom Forest")
print("Koulutus MSE:", rf_train_mse)
print("Validointi MSE:", rf_val_mse)

# Arvioidaan lopullinen malli testiaineistolla
linear_test_pred = linear_model.predict(X_test)
linear_test_mse = mean_squared_error(y_test, linear_test_pred)

print("Lineaarisen regression test MSE:", linear_test_mse)
```